[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/05_continuous_distributions/exercises.ipynb)

# Exercises — Module 05 — Continuous Distributions

20 fully solved problems in four tiers: L0 Concept Checks (4), L1 Foundations (6), L2 Applications in AI/ML and Physics (6), L3 Challenge Proofs (4).

Every numeric answer below is recomputed by a code cell that ran; nothing is quoted from memory.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

from math import e, exp, gamma as Gamma_fn, log, pi, sqrt
from scipy import integrate, optimize, stats
from scipy.special import erfc

print("preamble ready")

preamble ready


## L0 — Concept Checks

### Problem L0.1 — Does the Bus Care How Long You Have Waited?

**Statement.** Buses arrive with $\text{Exponential}(\lambda)$ gaps of mean 10 minutes. You have already waited 15 minutes. What is the expected additional wait, and what does the answer say about the model?

**Intuition.** An exponential survival function is a pure exponential, so dividing $S(15+t)$ by $S(15)$ cancels the elapsed time entirely.

**Solution**

*Step 1 — condition on survival.* With $S(t) = e^{-\lambda t}$ and $\lambda = 1/10$,

$$
P(X \gt 15 + t \mid X \gt 15) = \frac{e^{-\lambda(15+t)}}{e^{-\lambda\cdot 15}} = e^{-\lambda t} = P(X \gt t).
$$

*Step 2 — integrate the residual survival.* The remaining wait has exactly the original law, so $E[X - 15 \mid X \gt 15] = \int_0^\infty e^{-\lambda t}dt = 1/\lambda = 10$ minutes. The 15 minutes already spent are informationally worthless.

*Step 3 — read it as a modelling statement.* Constant hazard $h(t) = \lambda$ means the process never makes progress toward the next event. Real buses run on schedules with a hazard that rises as the scheduled time approaches, so the exponential model overstates the residual wait. The diagnostic is the hazard: if it is not flat, the law is not exponential.

$$
\boxed{E[\text{remaining wait}] = \frac{1}{\lambda} = 10 \text{ minutes, independent of the elapsed 15}}
$$

*Key takeaway:* Memorylessness is a strong modelling assumption, not a convenience — it asserts a system that never ages.

In [2]:
lam = 1 / 10
S = lambda t: np.exp(-lam * t)
for elapsed in (0, 5, 15, 60):
    resid = integrate.quad(lambda t: S(elapsed + t) / S(elapsed), 0, np.inf)[0]
    print(f"elapsed {elapsed:>3} min -> E[remaining] = {resid:.6f} min")
    assert abs(resid - 10.0) < 1e-8

sample = rng.exponential(1 / lam, size=2_000_000)
print(f"Monte Carlo E[X - 15 | X > 15] = {(sample[sample > 15] - 15).mean():.4f}  (exact 10)")

elapsed   0 min -> E[remaining] = 10.000000 min
elapsed   5 min -> E[remaining] = 10.000000 min
elapsed  15 min -> E[remaining] = 10.000000 min
elapsed  60 min -> E[remaining] = 10.000000 min
Monte Carlo E[X - 15 | X > 15] = 9.9912  (exact 10)


### Problem L0.2 — How Large Can a Density Be?

**Statement.** Give a density whose maximum value is $10^6$, and explain why a density's units make "is $f(x)$ big?" a meaningless question without context.

**Intuition.** A density is probability per unit of $x$, so squeezing a fixed unit of probability into a short interval makes the density as large as you like.

**Solution**

*Step 1 — construct one.* Take $X \sim \text{Unif}\left(0, 10^{-6}\right)$: then $f(x) = 10^{6}$ on the support and $\int f = 10^{6}\times10^{-6} = 1$. There is no upper bound at all, and unbounded densities are also legal — $\text{Beta}(0.5, 0.5)$ blows up at both endpoints while remaining integrable.

*Step 2 — check the units.* If $X$ is a time in seconds then $f_X$ carries units $\text{s}^{-1}$, so its numerical value changes with the unit. For $Y = 1000X$ (milliseconds),

$$
f_Y(y) = \frac{1}{1000}f_X\left(\frac{y}{1000}\right),
$$

so every density value is divided by 1000 while every probability is unchanged.

*Step 3 — draw the conclusion.* Only $\int_A f$ is dimensionless and unit-invariant. Hence "the density is large" is never a claim about probability; "the probability of this interval is large" is.

$$
\boxed{f \text{ is unbounded above; } f_X \text{ carries units } 1/[\,x\,] \text{ and rescales under a change of units}}
$$

*Key takeaway:* Densities are rates per unit of $x$; comparing density values across parameterizations or units is a category error.

In [3]:
width = 1e-6
print(f"Unif(0, 1e-6): peak density {1/width:.3e}, total mass {integrate.quad(lambda x: 1/width, 0, width)[0]:.12f}")
assert abs(integrate.quad(lambda x: 1 / width, 0, width)[0] - 1.0) < 1e-12

arcsine = stats.beta(0.5, 0.5)
print(f"Beta(0.5,0.5) density at x = 1e-6, 1e-10: {arcsine.pdf(1e-6):.3e}, {arcsine.pdf(1e-10):.3e} (unbounded)")
print(f"Beta(0.5,0.5) total mass {integrate.quad(arcsine.pdf, 0, 1)[0]:.12f} (still 1)")

sec = stats.expon(scale=2.0)                  # seconds
ms = stats.expon(scale=2000.0)                # the same variable in milliseconds
print(f"density at the mean:  seconds {sec.pdf(2.0):.6e} 1/s   milliseconds {ms.pdf(2000.0):.6e} 1/ms   ratio {sec.pdf(2.0)/ms.pdf(2000.0):.1f}")
print(f"probability of the same event: {sec.sf(2.0):.10f} vs {ms.sf(2000.0):.10f} (identical)")
assert abs(sec.sf(2.0) - ms.sf(2000.0)) < 1e-14

Unif(0, 1e-6): peak density 1.000e+06, total mass 1.000000000000
Beta(0.5,0.5) density at x = 1e-6, 1e-10: 3.183e+02, 3.183e+04 (unbounded)
Beta(0.5,0.5) total mass 1.000000000000 (still 1)
density at the mean:  seconds 1.839397e-01 1/s   milliseconds 1.839397e-04 1/ms   ratio 1000.0
probability of the same event: 0.3678794412 vs 0.3678794412 (identical)


### Problem L0.3 — A Distribution With No Mean

**Statement.** For Student's $t_\nu$, determine which moments exist, and evaluate the cases $\nu = 1, 2, 3$.

**Intuition.** The density decays like a power of $\lvert x\rvert$, so a moment exists exactly when the corresponding power integral converges at infinity.

**Solution**

*Step 1 — extract the tail exponent.* $f(x) \propto \left(1 + x^2/\nu\right)^{-(\nu+1)/2} \sim c\,\lvert x\rvert^{-(\nu+1)}$ for large $\lvert x\rvert$.

*Step 2 — test convergence.* Therefore

$$
E\left[\lvert X\rvert^{p}\right] \approx 2c\int_1^{\infty}x^{p}x^{-(\nu+1)}dx = 2c\int_1^{\infty}x^{p-\nu-1}dx,
$$

which converges if and only if $p - \nu - 1 \lt -1$, that is **if and only if $p \lt \nu$**.

*Step 3 — specialize.*

- $\nu = 1$ (Cauchy): no mean. The average of $n$ i.i.d. Cauchy variables is again Cauchy, so averaging accomplishes nothing and the law of large numbers does not apply.
- $\nu = 2$: the mean exists and equals $0$, the variance does not; the classical central limit theorem fails.
- $\nu = 3$: mean and variance exist, $\mathrm{Var} = \frac{\nu}{\nu-2} = 3$, but skewness and kurtosis (needing $p = 3, 4$) do not.

$$
\boxed{E\left[\lvert X\rvert^{p}\right] \lt \infty \iff p \lt \nu; \quad t_1 \text{ has no mean}, \; t_2 \text{ no variance}, \; t_3 \text{ no kurtosis}}
$$

*Key takeaway:* Heavy tails are about which moments exist, and non-existence breaks the tools — LLN, CLT, squared-error loss — that quietly assume them.

In [4]:
def truncated_abs_moment(nu, p, M):
    """2 * int_0^M x^p f_t(x) dx, computed in log space so the heavy tail stays accurate."""
    head = integrate.quad(lambda x: x**p * stats.t(nu).pdf(x), 0, 1, limit=200)[0]
    tail = integrate.quad(lambda u: np.exp((p + 1) * u) * stats.t(nu).pdf(np.exp(u)), 0, log(M), limit=400)[0]
    return 2 * (head + tail)


print(f"{'nu':>3} {'p':>4} {'M=1e2':>14} {'M=1e6':>16} {'M=1e12':>18}   verdict")
for nu in (1, 2, 3):
    for p in (0.5, 1.0, 2.0, 3.0):
        v2, v6, v12 = (truncated_abs_moment(nu, p, M) for M in (1e2, 1e6, 1e12))
        verdict = "converges (p < nu)" if p < nu else "diverges  (p >= nu)"
        print(f"{nu:>3} {p:>4} {v2:>14.4f} {v6:>16.4f} {v12:>18.4f}   {verdict}")
        if p < nu:
            assert abs(v12 - v6) < 0.01 * max(1.0, v6)
        else:
            assert v12 > 1.9 * v6
print(f"Var(t_3) = nu/(nu-2) = {3/(3-2):.4f}   scipy {stats.t(3).var():.4f}")

running = np.cumsum(rng.standard_cauchy(200_000)) / np.arange(1, 200_001)
print(f"Cauchy running mean at n = 1e3, 1e4, 1e5, 2e5: {running[[999, 9_999, 99_999, 199_999]]}")
print("The running mean of Cauchy data does not settle: it is Cauchy at every n.")

 nu    p          M=1e2            M=1e6             M=1e12   verdict
  1  0.5         1.2869           1.4129             1.4142   converges (p < nu)
  1  1.0         2.9318           8.7952            17.5905   diverges  (p >= nu)
  1  2.0        62.6683      636618.7724  636619772366.5791   diverges  (p >= nu)
  1  3.0      3180.1671 318309886174.9948 318309886183790081998848.0000   diverges  (p >= nu)
  2  0.5         1.0062           1.0075             1.0075   converges (p < nu)
  2  1.0         1.3942           1.4142             1.4142   converges (p < nu)
  2  2.0         7.9038          26.3242            53.9552   diverges  (p >= nu)
  2  3.0       194.4031     1999994.3432 1999999999994.3433   diverges  (p >= nu)
  3  0.5         0.9306           0.9306             0.9306   converges (p < nu)
  3  1.0         1.1023           1.1027             1.1027   converges (p < nu)
  3  2.0         2.9339           3.0000             3.0000   converges (p < nu)
  3  3.0        23.527

Cauchy running mean at n = 1e3, 1e4, 1e5, 2e5: [ 0.4515  0.149  -0.9377  0.0412]
The running mean of Cauchy data does not settle: it is Cauchy at every n.


### Problem L0.4 — Is a Flat Prior Uninformative?

**Statement.** You place a $\text{Unif}(0, 10)$ prior on a standard deviation $\sigma$. What prior does this induce on the variance $\sigma^2$, and on the precision $\tau = 1/\sigma^2$?

**Intuition.** Densities pick up a Jacobian under reparameterization, so flatness in one coordinate is a statement about that coordinate and nothing else.

**Solution**

*Step 1 — change variables to the variance.* With $\pi_\sigma(s) = 1/10$ on $(0,10)$ and $v = s^2$, so $s = \sqrt{v}$ and $\frac{ds}{dv} = \frac{1}{2\sqrt v}$,

$$
\pi_V(v) = \pi_\sigma\left(\sqrt v\right)\left\lvert \frac{ds}{dv}\right\rvert = \frac{1}{20\sqrt v}, \qquad 0 \lt v \lt 100 .
$$

Far from flat: a mild singularity at $v = 0$ favours small variances.

*Step 2 — change variables again to the precision.* For $\tau = 1/v$ we have $\left\lvert \frac{dv}{d\tau}\right\rvert = \tau^{-2}$, so

$$
\pi_T(\tau) = \frac{1}{20\sqrt{1/\tau}}\cdot\frac{1}{\tau^{2}} = \frac{1}{20}\tau^{-3/2}, \qquad \tau \gt 0.01 ,
$$

strongly concentrated near the lower boundary.

*Step 3 — conclude.* "Uniform" describes a *parameterization*, not a state of ignorance. The reparameterization-invariant construction is Jeffreys' prior $\pi(\theta) \propto \sqrt{\det I(\theta)}$, which for a scale parameter gives $\pi(\sigma) \propto 1/\sigma$ — equivalently uniform on $\ln\sigma$, hence identical whether one parameterizes by $\sigma$, $\sigma^2$ or $\tau$.

$$
\boxed{\pi_\sigma \text{ flat} \implies \pi_{\sigma^2}(v) \propto v^{-1/2}, \; \pi_\tau(\tau) \propto \tau^{-3/2}: \text{ flatness is parameterization-dependent}}
$$

*Key takeaway:* Densities transform with a Jacobian, so "non-informative" must be defined invariantly (Jeffreys), not by picking whichever coordinate looks flat.

In [5]:
sigma = rng.uniform(0, 10, size=2_000_000)
v_s, tau_s = sigma**2, 1 / sigma**2

pi_V = lambda v: 1 / (20 * np.sqrt(v))
pi_T = lambda t: t ** (-1.5) / 20

print(f"mass of pi_V on (0,100)      = {integrate.quad(pi_V, 0, 100)[0]:.12f}")
print(f"mass of pi_T on (0.01, inf)  = {integrate.quad(pi_T, 0.01, np.inf)[0]:.12f}")
assert abs(integrate.quad(pi_V, 0, 100)[0] - 1) < 1e-9 and abs(integrate.quad(pi_T, 0.01, np.inf)[0] - 1) < 1e-9

for lo, hi in [(0.0, 1.0), (1.0, 25.0), (25.0, 100.0)]:
    print(f"P(sigma^2 in [{lo:>5}, {hi:>5}]): simulated {np.mean((v_s >= lo) & (v_s < hi)):.5f}   formula {integrate.quad(pi_V, max(lo,1e-12), hi)[0]:.5f}")
print(f"P(tau < 0.1): simulated {np.mean(tau_s < 0.1):.5f}   formula {integrate.quad(pi_T, 0.01, 0.1)[0]:.5f}")

mass of pi_V on (0,100)      = 1.000000000000
mass of pi_T on (0.01, inf)  = 1.000000000000
P(sigma^2 in [  0.0,   1.0]): simulated 0.09998   formula 0.10000
P(sigma^2 in [  1.0,  25.0]): simulated 0.40018   formula 0.40000
P(sigma^2 in [ 25.0, 100.0]): simulated 0.49984   formula 0.50000
P(tau < 0.1): simulated 0.68363   formula 0.68377


## L1 — Foundations

### Problem L1.1 — Normalize, Integrate, and Invert a Density

**Statement.** Let $f(x) = c\,x e^{-x^2/2}$ for $x \gt 0$ (the Rayleigh density). Find $c$, the CDF, the median, and $E[X]$.

**Intuition.** The factor $x$ next to $e^{-x^2/2}$ is exactly the derivative of $x^2/2$, so one substitution turns every Rayleigh integral into an exponential one.

**Solution**

*Step 1 — normalize.* Substitute $u = x^2/2$, $du = x\,dx$:

$$
\int_0^{\infty}xe^{-x^2/2}dx = \int_0^{\infty}e^{-u}du = 1 \implies c = 1 .
$$

*Step 2 — integrate to the CDF.* The same substitution gives

$$
F(x) = \int_0^{x}te^{-t^2/2}dt = 1 - e^{-x^2/2}, \qquad x \ge 0 .
$$

*Step 3 — invert for the median.* Solve $1 - e^{-m^2/2} = \tfrac12$: $m^2 = 2\ln 2$, so $m = \sqrt{2\ln 2} \approx 1.1774$.

*Step 4 — integrate by parts for the mean.* With $u = x$ and $dv = xe^{-x^2/2}dx$, so $v = -e^{-x^2/2}$,

$$
E[X] = \int_0^{\infty}x^2e^{-x^2/2}dx = \left[-xe^{-x^2/2}\right]_0^{\infty} + \int_0^{\infty}e^{-x^2/2}dx = 0 + \frac{\sqrt{2\pi}}{2} = \sqrt{\frac{\pi}{2}} \approx 1.2533 ,
$$

using $\int_{-\infty}^{\infty}e^{-x^2/2}dx = \sqrt{2\pi}$ and symmetry. The mean exceeds the median, as expected for a right-skewed law. Note also $X^2 \sim \text{Exponential}(1/2) = \chi^2_2$, so the quantiles could have been read off the exponential directly.

$$
\boxed{c = 1, \quad F(x) = 1 - e^{-x^2/2}, \quad \text{median} = \sqrt{2\ln 2} \approx 1.177, \quad E[X] = \sqrt{\pi/2} \approx 1.253}
$$

*Key takeaway:* The substitution $u = x^2/2$ turns every Rayleigh computation into an exponential one — recognizing $X^2 \sim \chi^2_2$ is the structural version of the same shortcut.

In [6]:
f_ray = lambda x: x * np.exp(-x**2 / 2)
mass = integrate.quad(f_ray, 0, np.inf)[0]
median = sqrt(2 * log(2))
mean_ray = integrate.quad(lambda x: x * f_ray(x), 0, np.inf)[0]

print(f"c: total mass of x e^(-x^2/2) on (0, inf) = {mass:.12f}  ->  c = 1")
print(f"median: sqrt(2 ln 2) = {median:.6f}   F(median) = {1 - exp(-median**2/2):.12f}")
print(f"mean: quadrature {mean_ray:.10f}   closed form sqrt(pi/2) = {sqrt(pi/2):.10f}")
assert abs(mass - 1) < 1e-10 and abs(mean_ray - sqrt(pi / 2)) < 1e-10

ray = rng.rayleigh(1.0, size=1_000_000)
print(f"Monte Carlo median {np.median(ray):.4f}, mean {ray.mean():.4f}")
print(f"X^2 vs Exponential(1/2): KS p = {stats.kstest(ray**2, stats.expon(scale=2).cdf).pvalue:.3f}")

c: total mass of x e^(-x^2/2) on (0, inf) = 1.000000000000  ->  c = 1
median: sqrt(2 ln 2) = 1.177410   F(median) = 0.500000000000
mean: quadrature 1.2533141373   closed form sqrt(pi/2) = 1.2533141373
Monte Carlo median 1.1775, mean 1.2527
X^2 vs Exponential(1/2): KS p = 0.457


### Problem L1.2 — Exponential Hazard vs Weibull Hazard

**Statement.** A component has survival $S(t) = e^{-(t/5)^{2}}$ (years). Compute the hazard, the median life, and compare the failure rate at $t = 1$ and $t = 8$ with an exponential of the same median.

**Intuition.** Matching one summary such as the median leaves the hazard shape completely free, and the hazard is what governs maintenance schedules.

**Solution**

*Step 1 — differentiate the log survival.* This is Weibull with shape $k = 2$ and scale $\lambda = 5$:

$$
h(t) = -\frac{d}{dt}\ln S(t) = -\frac{d}{dt}\left(-\frac{t^2}{25}\right) = \frac{2t}{25},
$$

a hazard growing linearly — a wear-out component.

*Step 2 — invert for the median.* $e^{-(m/5)^2} = \tfrac12$ gives $(m/5)^2 = \ln 2$, so $m = 5\sqrt{\ln 2} \approx 4.1628$ years.

*Step 3 — match an exponential and compare.* An exponential with the same median has $e^{-\lambda m} = \tfrac12$, so $\lambda = \ln 2/4.1628 \approx 0.16651$ per year, flat forever. The Weibull hazards are

$$
h(1) = \frac{2}{25} = 0.08, \qquad h(8) = \frac{16}{25} = 0.64 .
$$

At one year the true component is roughly *half* as likely to fail as the exponential model claims; at eight years it is nearly **four times** as likely. Fitting an exponential therefore over-maintains new units and badly under-predicts end-of-life failures.

$$
\boxed{h(t) = \frac{2t}{25}, \quad \text{median} \approx 4.163 \text{ yr}, \quad h(1) = 0.08 \lt 0.1665 \lt h(8) = 0.64}
$$

*Key takeaway:* Matching a single summary says nothing about the hazard shape, and the hazard is what drives maintenance and warranty decisions.

In [7]:
S_w = lambda t: np.exp(-((t / 5) ** 2))
h_w = lambda t: 2 * t / 25
median_w = 5 * sqrt(log(2))
lam_exp_match = log(2) / median_w

num_h = lambda t: (-(S_w(t + 1e-6) - S_w(t - 1e-6)) / 2e-6) / S_w(t)
print(f"median  {median_w:.6f} yr   S(median) = {S_w(median_w):.12f}")
print(f"matched exponential rate  {lam_exp_match:.6f} per yr")
for t in (1.0, 8.0):
    print(f"h({t:.0f}) formula {h_w(t):.4f}   numerical derivative {num_h(t):.4f}   exponential {lam_exp_match:.4f}")
    assert abs(h_w(t) - num_h(t)) < 1e-5
assert abs(S_w(median_w) - 0.5) < 1e-12
print(f"ratio h(8)/lambda = {h_w(8)/lam_exp_match:.3f},  h(1)/lambda = {h_w(1)/lam_exp_match:.3f}")

median  4.162773 yr   S(median) = 0.500000000000
matched exponential rate  0.166511 per yr
h(1) formula 0.0800   numerical derivative 0.0800   exponential 0.1665
h(8) formula 0.6400   numerical derivative 0.6400   exponential 0.1665
ratio h(8)/lambda = 3.844,  h(1)/lambda = 0.480


### Problem L1.3 — Sums of Exponentials Are Gamma

**Statement.** Server requests pass through 3 independent stages, each $\text{Exponential}(\lambda = 2\ \text{s}^{-1})$. Find the density of the total latency $T$, its mean and variance, and $P(T \gt 3)$.

**Intuition.** Independent exponentials with a common rate add their shape parameters, so the total is Erlang and its tail is a truncated Poisson sum.

**Solution**

*Step 1 — identify the law.* By Gamma additivity, $T \sim \text{Gamma}(3,2)$ (Erlang-3), so with $\Gamma(3) = 2! = 2$,

$$
f_T(t) = \frac{2^{3}}{\Gamma(3)}t^{2}e^{-2t} = 4t^2e^{-2t}, \qquad t \gt 0 .
$$

*Step 2 — read off the moments.*

$$
E[T] = \frac{\alpha}{\lambda} = 1.5\ \text{s}, \qquad \mathrm{Var}(T) = \frac{\alpha}{\lambda^2} = 0.75\ \text{s}^2, \qquad \mathrm{sd} \approx 0.866\ \text{s} .
$$

*Step 3 — use the Erlang-Poisson duality for the tail.* With $\lambda t = 6$,

$$
P(T \gt 3) = \sum_{j=0}^{2}\frac{6^{j}e^{-6}}{j!} = e^{-6}\left(1 + 6 + 18\right) = 25e^{-6} = 0.061969 .
$$

A single $\text{Exponential}(2/3)$ with the same mean would have variance $2.25$, three times larger: staging *concentrates* latency, which is why multi-stage pipelines have more predictable tails than a single memoryless step.

$$
\boxed{f_T(t) = 4t^2e^{-2t}, \quad E[T] = 1.5, \quad \mathrm{Var}(T) = 0.75, \quad P(T \gt 3) = 25e^{-6} \approx 0.06197}
$$

*Key takeaway:* The Erlang survival function is a truncated Poisson sum — the same mechanism describes counts and waiting times simultaneously.

In [8]:
erlang = stats.gamma(3.0, scale=0.5)
tail_closed = 25 * exp(-6)
tail_quad = integrate.quad(lambda t: 4 * t**2 * np.exp(-2 * t), 3, np.inf)[0]
tail_poisson = sum(6.0**j * exp(-6) / Gamma_fn(j + 1) for j in range(3))

print(f"E[T] {erlang.mean():.6f} (exact 1.5)   Var[T] {erlang.var():.6f} (exact 0.75)   sd {erlang.std():.6f}")
print(f"P(T>3): quadrature {tail_quad:.10f}   Poisson sum {tail_poisson:.10f}   25 e^-6 {tail_closed:.10f}   scipy {erlang.sf(3):.10f}")
assert max(abs(tail_quad - tail_closed), abs(tail_poisson - tail_closed), abs(erlang.sf(3) - tail_closed)) < 1e-10

one_stage = stats.expon(scale=1.5)
print(f"variance of a single Exponential with the same mean: {one_stage.var():.4f} (three times 0.75)")

E[T] 1.500000 (exact 1.5)   Var[T] 0.750000 (exact 0.75)   sd 0.866025
P(T>3): quadrature 0.0619688044   Poisson sum 0.0619688044   25 e^-6 0.0619688044   scipy 0.0619688044
variance of a single Exponential with the same mean: 2.2500 (three times 0.75)


### Problem L1.4 — Standardization and Gaussian Tail Probabilities

**Statement.** For $X \sim \mathcal{N}(100, 15^2)$ compute $P(X \gt 130)$, $P(85 \lt X \lt 115)$, and the 95th percentile. Then explain why $P(X \gt 190)$ should not be computed as $1 - \Phi(6)$ in floating point.

**Intuition.** Standardizing reduces every Gaussian question to $\Phi$, but $1-\Phi$ subtracts two nearly equal numbers once you are far out in the tail.

**Solution**

*Step 1 — standardize.* Put $Z = (X-100)/15$.

$$
P(X \gt 130) = P(Z \gt 2) = 1 - \Phi(2) = 0.022750 .
$$

$$
P(85 \lt X \lt 115) = P(-1 \lt Z \lt 1) = 2\Phi(1) - 1 = 2(0.841345) - 1 = 0.682689 .
$$

*Step 2 — invert for the percentile.* $z_{0.95} = 1.644854$, so $x_{0.95} = 100 + 15(1.644854) = 124.673$.

*Step 3 — diagnose the floating-point failure.* $P(X \gt 190) = P(Z \gt 6) \approx 9.866\times10^{-10}$. Computing it as $1 - \Phi(6)$ subtracts two doubles agreeing to about ten digits, and cancellation destroys most of the significant digits; by $z \approx 8$ the naive value is pure rounding noise and by $z \approx 9$ it is exactly zero. The stable route is

$$
P(Z \gt z) = \tfrac12\operatorname{erfc}\left(\frac{z}{\sqrt2}\right),
$$

evaluated directly in the tail and accurate down to $10^{-300}$. Libraries expose it as the survival function `sf`.

$$
\boxed{P(X \gt 130) = 0.02275, \; P(85 \lt X \lt 115) = 0.68269, \; x_{0.95} \approx 124.67, \; P(X \gt 190) \approx 9.866\times10^{-10}}
$$

*Key takeaway:* Standardization reduces every Gaussian question to $\Phi$; far in the tail use `erfc` or `sf` rather than $1 - \Phi$.

In [9]:
X = stats.norm(100, 15)
print(f"P(X > 130)        = {X.sf(130):.6f}")
print(f"P(85 < X < 115)   = {X.cdf(115) - X.cdf(85):.6f}")
print(f"95th percentile   = {X.ppf(0.95):.4f}   (100 + 15 * {stats.norm.ppf(0.95):.6f})")
print(f"P(X > 190)        = {X.sf(190):.6e}")
assert abs(X.sf(130) - 0.022750) < 1e-6 and abs((X.cdf(115) - X.cdf(85)) - 0.682689) < 1e-6

print("\n  z    naive 1 - Phi(z)      stable 0.5*erfc(z/sqrt2)   relative error")
for z in (2.0, 6.0, 8.0, 9.0, 12.0):
    naive = 1 - stats.norm.cdf(z)
    stable = 0.5 * erfc(z / sqrt(2))
    print(f"{z:5.1f}  {naive:.12e}   {stable:.12e}   {abs(naive-stable)/stable:.2e}")

P(X > 130)        = 0.022750
P(85 < X < 115)   = 0.682689
95th percentile   = 124.6728   (100 + 15 * 1.644854)
P(X > 190)        = 9.865876e-10

  z    naive 1 - Phi(z)      stable 0.5*erfc(z/sqrt2)   relative error
  2.0  2.275013194818e-02   2.275013194818e-02   3.05e-16
  6.0  9.865877004245e-10   9.865876450377e-10   5.61e-08
  8.0  6.661338147751e-16   6.220960574272e-16   7.08e-02
  9.0  0.000000000000e+00   1.128588405954e-19   1.00e+00
 12.0  0.000000000000e+00   1.776482112078e-33   1.00e+00


### Problem L1.5 — Beta Shapes and the Effect of Evidence

**Statement.** Start from a $\text{Beta}(1,1)$ prior on a click-through rate $p$ and observe 7 clicks in 10 impressions. Give the posterior, its mean, mode and standard deviation, and compare with the MLE.

**Intuition.** Beta parameters are pseudo-counts, so a Bernoulli likelihood updates them by plain addition.

**Solution**

*Step 1 — multiply prior by likelihood.* With $s = 7$ successes and $n - s = 3$ failures,

$$
\pi(p \mid \text{data}) \propto p^{s}(1-p)^{n-s}\cdot p^{\alpha-1}(1-p)^{\beta-1} = p^{7}(1-p)^{3} \implies \text{Beta}(8,4) .
$$

*Step 2 — read off the summaries* with $\alpha = 8$, $\beta = 4$:

$$
E[p] = \frac{\alpha}{\alpha+\beta} = \frac{8}{12} = 0.6667, \qquad \text{mode} = \frac{\alpha-1}{\alpha+\beta-2} = \frac{7}{10} = 0.70,
$$

$$
\mathrm{Var}(p) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)} = \frac{32}{144\times13} = 0.017094, \qquad \mathrm{sd} = 0.13074 .
$$

*Step 3 — compare with the MLE.* The posterior mode equals the MLE $\hat p = 0.7$ because a $\text{Beta}(1,1)$ prior is flat, so MAP $=$ MLE here. The posterior *mean* is pulled toward $0.5$: the Laplace add-one rule $\frac{s+1}{n+2}$ is exactly this posterior mean. The standard deviation $0.131$ is the honest statement that 10 impressions pin the rate only to about $\pm 0.26$ at two sigma.

$$
\boxed{p \mid \text{data} \sim \text{Beta}(8,4); \; E[p] = \tfrac23, \; \text{mode} = 0.7 = \hat p_{\text{MLE}}, \; \mathrm{sd} \approx 0.1307}
$$

*Key takeaway:* Beta parameters are pseudo-counts; conjugacy turns Bayesian updating into addition, and add-one smoothing is a posterior mean in disguise.

In [10]:
post = stats.beta(8, 4)
grid = np.linspace(1e-9, 1 - 1e-9, 200_001)
unnorm = grid**7 * (1 - grid) ** 3
numeric_mean = np.trapezoid(grid * unnorm, grid) / np.trapezoid(unnorm, grid)

print(f"posterior mean  closed {8/12:.6f}   from the raw likelihood x prior {numeric_mean:.6f}   scipy {post.mean():.6f}")
print(f"posterior mode  {(8-1)/(8+4-2):.6f}   MLE 7/10 = {7/10:.6f}")
print(f"posterior var   {post.var():.6f}   sd {post.std():.6f}")
print(f"add-one smoothing (s+1)/(n+2) = {(7+1)/(10+2):.6f}  equals the posterior mean")
assert abs(numeric_mean - 8 / 12) < 1e-6 and abs(post.std() - 0.130744) < 1e-5

posterior mean  closed 0.666667   from the raw likelihood x prior 0.666667   scipy 0.666667
posterior mode  0.700000   MLE 7/10 = 0.700000
posterior var   0.017094   sd 0.130744
add-one smoothing (s+1)/(n+2) = 0.666667  equals the posterior mean


### Problem L1.6 — Lognormal Mean vs Median

**Statement.** Let $Y \sim \mathcal{N}(\mu,\sigma^2)$ and $X = e^{Y}$ with $\mu = 0$, $\sigma = 1$. Derive the density of $X$, compute its mean, median and mode, and explain the ordering.

**Intuition.** Exponentiation is monotone, so quantiles transport unchanged while the mean picks up a Jensen penalty.

**Solution**

*Step 1 — transform the density.* $g(y) = e^y$ is strictly increasing with $g^{-1}(x) = \ln x$ and $\frac{d}{dx}\ln x = 1/x$, so

$$
f_X(x) = f_Y(\ln x)\cdot\frac1x = \frac{1}{x\sqrt{2\pi}}\exp\left(-\frac{(\ln x)^2}{2}\right), \qquad x \gt 0 .
$$

*Step 2 — mean by the Gaussian MGF at $t = 1$.* $E[X] = E\left[e^{Y}\right] = M_Y(1) = e^{\mu+\sigma^2/2} = e^{0.5} \approx 1.64872$.

*Step 3 — median by monotone transport.* $\operatorname{med}(X) = e^{\operatorname{med}(Y)} = e^{\mu} = 1$.

*Step 4 — mode by differentiating the log density.* $\frac{d}{dx}\ln f_X = -\frac1x - \frac{\ln x}{x} = 0$ gives $\ln x = -1$, so the mode is $e^{\mu-\sigma^2} = e^{-1} \approx 0.36788$.

*Step 5 — interpret the ordering.* mode $\lt$ median $\lt$ mean is the signature of right skew. The mean exceeds the median by $e^{\sigma^2/2}$, which explodes with $\sigma$: at $\sigma = 2$ the mean is $e^{2} \approx 7.4$ times the median. This is why "average" and "typical" diverge for incomes, latencies and file sizes.

$$
\boxed{\text{mode} = e^{\mu-\sigma^2} \lt \text{median} = e^{\mu} \lt \text{mean} = e^{\mu+\sigma^2/2}}
$$

*Key takeaway:* Under a nonlinear transform quantiles pass through unchanged but means do not — $E[g(X)] \ne g(E[X])$ is Jensen's inequality in action.

In [11]:
LN = stats.lognorm(1.0)                       # mu = 0, sigma = 1
f_ln = lambda x: np.exp(-(np.log(x) ** 2) / 2) / (x * sqrt(2 * pi))
mode_num = optimize.minimize_scalar(lambda x: -f_ln(x), bounds=(1e-6, 5), method="bounded").x

print(f"mean    quadrature {integrate.quad(lambda x: x*f_ln(x), 0, np.inf)[0]:.8f}   closed e^0.5 = {exp(0.5):.8f}")
print(f"median  scipy      {LN.median():.8f}                 closed e^0    = {1.0:.8f}")
print(f"mode    numerical  {mode_num:.8f}                 closed e^-1   = {exp(-1):.8f}")
assert abs(integrate.quad(lambda x: x * f_ln(x), 0, np.inf)[0] - exp(0.5)) < 1e-8
assert abs(mode_num - exp(-1)) < 1e-5

for sig in (0.5, 1.0, 2.0):
    law = stats.lognorm(sig)
    print(f"sigma = {sig}: mean/median = {law.mean()/law.median():.5f}   e^(sigma^2/2) = {exp(sig**2/2):.5f}")

mean    quadrature 1.64872127   closed e^0.5 = 1.64872127
median  scipy      1.00000000                 closed e^0    = 1.00000000
mode    numerical  0.36787913                 closed e^-1   = 0.36787944
sigma = 0.5: mean/median = 1.13315   e^(sigma^2/2) = 1.13315
sigma = 1.0: mean/median = 1.64872   e^(sigma^2/2) = 1.64872
sigma = 2.0: mean/median = 7.38906   e^(sigma^2/2) = 7.38906


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Squared-Error Loss Is a Gaussian Log-Likelihood

**Statement.** For a regression model $y_i = f_\theta(x_i) + \varepsilon_i$ with $\varepsilon_i \sim \mathcal{N}(0,\sigma^2)$ i.i.d., derive the negative log-likelihood and show that minimizing it over $\theta$ is least squares. Then repeat with Laplace noise.

**Intuition.** Taking $-\ln$ of a product of densities turns the exponent of the noise density directly into the loss.

**Solution**

*Step 1 — write the Gaussian negative log-likelihood.* The likelihood is $\prod_{i=1}^n \frac{1}{\sigma\sqrt{2\pi}}\exp\left(-\frac{\left(y_i-f_\theta(x_i)\right)^2}{2\sigma^2}\right)$, so

$$
-\ln L(\theta,\sigma) = \frac{n}{2}\ln\left(2\pi\sigma^2\right) + \frac{1}{2\sigma^2}\sum_{i=1}^n\left(y_i - f_\theta(x_i)\right)^2 .
$$

*Step 2 — drop the $\theta$-free terms.* The first term does not involve $\theta$ and the second is a positive multiple of the residual sum of squares, so

$$
\arg\min_\theta\left[-\ln L\right] = \arg\min_\theta \sum_{i=1}^n\left(y_i-f_\theta(x_i)\right)^2 .
$$

Profiling out $\sigma$ gives $\hat\sigma^2 = \frac1n\sum_i r_i^2$, the mean squared residual.

*Step 3 — repeat with Laplace noise.* With $\varepsilon_i \sim \text{Laplace}(0,b)$, $f(\varepsilon) = \frac{1}{2b}e^{-\lvert\varepsilon\rvert/b}$, so

$$
-\ln L(\theta,b) = n\ln(2b) + \frac1b\sum_{i=1}^n\left\lvert y_i - f_\theta(x_i)\right\rvert ,
$$

and the MLE minimizes the sum of *absolute* residuals — median regression, whose influence function is bounded and therefore robust. The Huber loss interpolates: quadratic near $0$ (Gaussian core, efficient) and linear in the tails (Laplace, robust).

$$
\boxed{\text{Gaussian noise} \iff L_2 \text{ loss}; \qquad \text{Laplace noise} \iff L_1 \text{ loss}}
$$

*Key takeaway:* A loss function is a noise-model declaration; if residuals are heavy-tailed, an $L_2$ loss is not merely suboptimal but statistically misspecified.

In [12]:
n_obs = 200
x_obs = rng.uniform(-2, 2, n_obs)
design = np.column_stack([np.ones(n_obs), x_obs])
theta_true = np.array([1.0, -0.5])
y_obs = design @ theta_true + rng.normal(0, 0.7, n_obs)
y_obs[:5] += 25.0                                            # five gross outliers

nll_gauss = lambda th: 0.5 * np.sum((y_obs - design @ th) ** 2)
nll_lap = lambda th: np.sum(np.abs(y_obs - design @ th))

theta_ls = np.linalg.lstsq(design, y_obs, rcond=None)[0]
theta_gauss = optimize.minimize(nll_gauss, np.zeros(2)).x
theta_lap = optimize.minimize(nll_lap, np.zeros(2), method="Nelder-Mead", options={"xatol": 1e-10, "fatol": 1e-12, "maxiter": 20000}).x

print(f"least squares (lstsq)          {theta_ls}")
print(f"argmin of Gaussian NLL         {theta_gauss}")
print(f"argmin of Laplace NLL (L1)     {theta_lap}")
print(f"true parameters                {theta_true}")
assert np.allclose(theta_ls, theta_gauss, atol=1e-6)
print(f"L2 fit error {np.linalg.norm(theta_ls - theta_true):.4f}  vs  L1 fit error {np.linalg.norm(theta_lap - theta_true):.4f} (outliers present)")
assert np.linalg.norm(theta_lap - theta_true) < np.linalg.norm(theta_ls - theta_true)

sigma_hat = sqrt(np.mean((y_obs - design @ theta_ls) ** 2))
print(f"profiled sigma-hat = sqrt(mean squared residual) = {sigma_hat:.4f}")

least squares (lstsq)          [ 1.5938 -0.2648]
argmin of Gaussian NLL         [ 1.5938 -0.2648]
argmin of Laplace NLL (L1)     [ 1.0099 -0.4407]
true parameters                [ 1.  -0.5]
L2 fit error 0.6387  vs  L1 fit error 0.0601 (outliers present)
profiled sigma-hat = sqrt(mean squared residual) = 3.9753


### Problem L2.2 — The Gaussian KL Term in a VAE

**Statement.** Derive $D_{\mathrm{KL}}\left(\mathcal{N}(\mu,\sigma^2)\,\parallel\,\mathcal{N}(0,1)\right)$ in closed form and interpret each term as a force acting on the encoder.

**Intuition.** Both log-densities are quadratics, so the divergence needs only the first two moments of $q$.

**Solution**

*Step 1 — expand the log-ratio.* Write $q = \mathcal{N}(\mu,\sigma^2)$ and $p = \mathcal{N}(0,1)$. The shared $-\tfrac12\ln(2\pi)$ cancels, leaving

$$
D_{\mathrm{KL}}(q \parallel p) = E_q\left[\ln q(X) - \ln p(X)\right] = E_q\left[-\frac{(X-\mu)^2}{2\sigma^2} - \ln\sigma + \frac{X^2}{2}\right] .
$$

*Step 2 — evaluate the two expectations.* $E_q\left[(X-\mu)^2\right] = \sigma^2$ and $E_q\left[X^2\right] = \mu^2+\sigma^2$, so

$$
D_{\mathrm{KL}}(q\parallel p) = -\frac12 - \ln\sigma + \frac{\mu^2+\sigma^2}{2} = \frac12\left(\mu^2 + \sigma^2 - \ln\sigma^2 - 1\right).
$$

*Step 3 — read the forces.* The $\mu^2$ term pulls posterior means toward the origin. The pair $\sigma^2 - \ln\sigma^2$ is minimized at $\sigma = 1$ (derivative $1 - 1/\sigma^2 = 0$), penalizing both over-confident encoders ($\sigma \to 0$, where $-\ln\sigma^2\to\infty$) and over-dispersed ones. The whole expression is $0$ exactly when $\mu = 0$, $\sigma = 1$.

*Step 4 — note the failure mode.* Because the minimum is attained *at the prior*, a decoder strong enough to ignore $z$ makes $\mu = 0$, $\sigma = 1$ optimal and the latent carries no information — posterior collapse, mitigated by KL annealing or free-bits floors.

$$
\boxed{D_{\mathrm{KL}}\left(\mathcal{N}(\mu,\sigma^2)\,\parallel\,\mathcal{N}(0,1)\right) = \frac12\left(\mu^2 + \sigma^2 - \ln\sigma^2 - 1\right)}
$$

*Key takeaway:* The VAE regularizer is a two-moment penalty in closed form — which is exactly why Gaussians are chosen for the encoder and the prior.

In [13]:
kl_closed = lambda mu, sig: 0.5 * (mu**2 + sig**2 - log(sig**2) - 1)


def kl_numeric(mu, sig):
    q, p = stats.norm(mu, sig), stats.norm(0, 1)
    return integrate.quad(lambda x: q.pdf(x) * (q.logpdf(x) - p.logpdf(x)), mu - 40 * sig, mu + 40 * sig, limit=400)[0]


for mu, sig in [(0.0, 1.0), (1.5, 1.0), (0.0, 0.25), (0.0, 3.0), (-2.0, 0.6)]:
    print(f"mu={mu:>5}, sigma={sig:<5}: closed {kl_closed(mu, sig):.10f}   numeric {kl_numeric(mu, sig):.10f}")
    assert abs(kl_closed(mu, sig) - kl_numeric(mu, sig)) < 1e-8

opt = optimize.minimize_scalar(lambda s: kl_closed(0.0, s), bounds=(0.05, 5), method="bounded")
print(f"KL is minimized at sigma = {opt.x:.8f} with value {opt.fun:.3e}  (the prior itself)")
assert abs(opt.x - 1.0) < 1e-5

mu=  0.0, sigma=1.0  : closed 0.0000000000   numeric 0.0000000000
mu=  1.5, sigma=1.0  : closed 1.1250000000   numeric 1.1250000000
mu=  0.0, sigma=0.25 : closed 0.9175443611   numeric 0.9175443611
mu=  0.0, sigma=3.0  : closed 2.9013877113   numeric 2.9013877113
mu= -2.0, sigma=0.6  : closed 2.1908256238   numeric 2.1908256238
KL is minimized at sigma = 0.99999843 with value 2.473e-12  (the prior itself)


### Problem L2.3 — Maxwell-Boltzmann Speeds From Gaussian Components

**Statement.** In an ideal gas each velocity component is i.i.d. $\mathcal{N}(0,\sigma^2)$ with $\sigma^2 = k_BT/m$. Derive the density of the speed $V = \lVert\mathbf{v}\rVert$, and find the most probable, mean and RMS speeds.

**Intuition.** The joint density depends on $\mathbf v$ only through its norm, so the density of the speed is the radial density times the area of the sphere of that radius.

**Solution**

*Step 1 — write the isotropic joint density.*

$$
f_{\mathbf v}(\mathbf v) = \left(2\pi\sigma^2\right)^{-3/2}\exp\left(-\frac{\lVert\mathbf v\rVert^2}{2\sigma^2}\right).
$$

*Step 2 — integrate over the sphere of radius $v$*, whose surface area is $4\pi v^2$:

$$
f_V(v) = 4\pi v^2\left(2\pi\sigma^2\right)^{-3/2}\exp\left(-\frac{v^2}{2\sigma^2}\right) = \sqrt{\frac{2}{\pi}}\,\frac{v^2}{\sigma^3}\exp\left(-\frac{v^2}{2\sigma^2}\right), \qquad v \gt 0 ,
$$

the Maxwell density: a $\chi_3$ law scaled by $\sigma$, equivalently $V^2/\sigma^2 \sim \chi^2_3$.

*Step 3 — extract the three speeds.* Differentiating $\ln f_V = 2\ln v - \frac{v^2}{2\sigma^2} + c$ gives $\frac2v = \frac{v}{\sigma^2}$, so $v_p = \sqrt2\,\sigma = \sqrt{2k_BT/m}$. The mean is $E[V] = \sigma E\left[\chi_3\right] = \sigma\sqrt{8/\pi} = \sqrt{8k_BT/(\pi m)}$, and $E[V^2] = 3\sigma^2$ gives $v_{\text{rms}} = \sqrt{3k_BT/m}$.

*Step 4 — order them.* $v_p \lt \bar v \lt v_{\text{rms}}$, with ratios $\sqrt2 \lt \sqrt{8/\pi} \lt \sqrt3$, i.e. $1.4142 \lt 1.5958 \lt 1.7321$ — a consequence of right skew plus Jensen's inequality.

$$
\boxed{f_V(v) \propto v^2 e^{-v^2/(2\sigma^2)}; \quad v_p = \sqrt{\tfrac{2k_BT}{m}} \lt \bar v = \sqrt{\tfrac{8k_BT}{\pi m}} \lt v_{\text{rms}} = \sqrt{\tfrac{3k_BT}{m}}}
$$

*Key takeaway:* The $v^2$ factor is the geometry of the sphere, not the physics — the same $\chi_k$ construction governs the norm of any isotropic Gaussian vector, including neural-network weight vectors.

In [14]:
sig_v = 1.0
f_V = lambda v: sqrt(2 / pi) * v**2 * np.exp(-(v**2) / (2 * sig_v**2)) / sig_v**3

mass_V = integrate.quad(f_V, 0, np.inf)[0]
v_p = optimize.minimize_scalar(lambda v: -f_V(v), bounds=(1e-6, 10), method="bounded").x
v_bar = integrate.quad(lambda v: v * f_V(v), 0, np.inf)[0]
v_rms = sqrt(integrate.quad(lambda v: v**2 * f_V(v), 0, np.inf)[0])

print(f"total mass {mass_V:.12f}")
print(f"most probable {v_p:.6f}  closed sqrt(2)     = {sqrt(2):.6f}")
print(f"mean          {v_bar:.6f}  closed sqrt(8/pi)  = {sqrt(8/pi):.6f}")
print(f"rms           {v_rms:.6f}  closed sqrt(3)     = {sqrt(3):.6f}")
assert abs(mass_V - 1) < 1e-9 and abs(v_p - sqrt(2)) < 1e-5
assert abs(v_bar - sqrt(8 / pi)) < 1e-8 and abs(v_rms - sqrt(3)) < 1e-8

speeds = np.linalg.norm(rng.standard_normal((500_000, 3)) * sig_v, axis=1)
print(f"Monte Carlo mean {speeds.mean():.4f}, rms {sqrt((speeds**2).mean()):.4f}")
print(f"KS speeds vs chi(3): p = {stats.kstest(speeds, stats.chi(3).cdf).pvalue:.3f}")

total mass 1.000000000000
most probable 1.414214  closed sqrt(2)     = 1.414214
mean          1.595769  closed sqrt(8/pi)  = 1.595769
rms           1.732051  closed sqrt(3)     = 1.732051
Monte Carlo mean 1.5949, rms 1.7315
KS speeds vs chi(3): p = 0.158


### Problem L2.4 — Energy Fluctuations and the Thermodynamic Limit

**Statement.** A monatomic ideal gas of $N$ molecules has $d = 3N$ velocity components, each i.i.d. $\mathcal{N}(0, k_BT/m)$. Show that the total kinetic energy satisfies $E = \tfrac12 k_BT\,\chi^2_d$, deduce equipartition $\langle E\rangle = \tfrac{d}{2}k_BT$ and the relative fluctuation $\sqrt{2/d}$, and state the geometric consequence for a standard Gaussian vector in $\mathbb{R}^d$.

**Intuition.** A sum of $d$ independent squares has mean $d$ and variance $2d$, so its relative spread shrinks like $d^{-1/2}$ — macroscopic quantities stop fluctuating.

**Solution**

*Step 1 — reduce to a chi-square.* Write $v_i = \sigma z_i$ with $\sigma^2 = k_BT/m$ and $z_i$ i.i.d. standard normal. Then

$$
E = \frac{m}{2}\sum_{i=1}^{d} v_i^2 = \frac{m\sigma^2}{2}\sum_{i=1}^{d} z_i^2 = \frac{k_BT}{2}\,\chi^2_d .
$$

*Step 2 — take moments.* Since $E\left[\chi^2_d\right] = d$ and $\mathrm{Var}\left(\chi^2_d\right) = 2d$,

$$
\langle E\rangle = \frac{d}{2}k_BT \quad (\text{equipartition: } \tfrac12 k_BT \text{ per degree of freedom}), \qquad \mathrm{Var}(E) = \frac{d}{2}\left(k_BT\right)^2 .
$$

*Step 3 — form the relative fluctuation.*

$$
\frac{\mathrm{sd}(E)}{\langle E\rangle} = \frac{\sqrt{d/2}\,k_BT}{(d/2)k_BT} = \sqrt{\frac{2}{d}} \xrightarrow[d\to\infty]{} 0 .
$$

For one mole, $d \approx 1.8\times10^{24}$ and the relative fluctuation is about $10^{-12}$: this is why thermodynamics can treat energy as a *number* rather than a random variable.

*Step 4 — read off the geometry.* The same computation with $\sigma = 1$ says $\lVert\mathbf z\rVert^2 \sim \chi^2_d$ with mean $d$ and variance $2d$, and a delta-method expansion of $\sqrt{u}$ at $u = d$ gives

$$
E\left[\lVert\mathbf z\rVert\right] \approx \sqrt d\left(1 - \frac{1}{4d}\right), \qquad \mathrm{Var}\left(\lVert\mathbf z\rVert\right) \approx \frac{2d}{4d} = \frac12 .
$$

The radius grows like $\sqrt d$ while its absolute spread stays $O(1)$: for $d = 1000$ the norm is $31.6 \pm 0.71$, a shell of relative thickness about $2\%$. Consequences: the density's mode is at the origin yet almost no sample is near it; two independent Gaussian vectors are nearly orthogonal; and Gaussian latent spaces are effectively spherical, which is why spherical interpolation between latents beats linear interpolation — the linear midpoint falls off the shell.

$$
\boxed{E = \tfrac12 k_BT\,\chi^2_d, \quad \langle E\rangle = \tfrac{d}{2}k_BT, \quad \frac{\mathrm{sd}(E)}{\langle E\rangle} = \sqrt{\tfrac2d}; \quad \lVert\mathbf z\rVert \approx \sqrt d \pm \tfrac{1}{\sqrt2}}
$$

*Key takeaway:* Equipartition and the high-dimensional "thin shell" are the same $\chi^2_d$ statement — concentration of measure is what makes both thermodynamics and latent-space geometry work.

In [15]:
for d in (3, 30, 3_000, 300_000):
    chi2 = stats.chi2(d)
    print(f"d={d:>7}: E[chi2] {chi2.mean():>10.1f}  Var {chi2.var():>12.1f}  relative fluctuation {chi2.std()/chi2.mean():.3e}  sqrt(2/d) {sqrt(2/d):.3e}")
    assert abs(chi2.std() / chi2.mean() - sqrt(2 / d)) < 1e-12

d = 1000
z = rng.standard_normal((200_000, d))
norms = np.linalg.norm(z, axis=1)
print(f"\nd = {d}: E||z|| simulated {norms.mean():.4f}   delta-method sqrt(d)(1 - 1/(4d)) = {sqrt(d)*(1-1/(4*d)):.4f}")
print(f"        Var||z|| simulated {norms.var():.4f}   delta-method 1/2 = 0.5")
assert abs(norms.mean() - sqrt(d) * (1 - 1 / (4 * d))) < 0.01 and abs(norms.var() - 0.5) < 0.02

cos_sim = np.sum(z[:100_000] * z[100_000:], axis=1) / (norms[:100_000] * norms[100_000:])
print(f"        cosine similarity of independent vectors: mean {cos_sim.mean():+.5f}, sd {cos_sim.std():.5f}, 1/sqrt(d) = {1/sqrt(d):.5f}")

d=      3: E[chi2]        3.0  Var          6.0  relative fluctuation 8.165e-01  sqrt(2/d) 8.165e-01
d=     30: E[chi2]       30.0  Var         60.0  relative fluctuation 2.582e-01  sqrt(2/d) 2.582e-01
d=   3000: E[chi2]     3000.0  Var       6000.0  relative fluctuation 2.582e-02  sqrt(2/d) 2.582e-02
d= 300000: E[chi2]   300000.0  Var     600000.0  relative fluctuation 2.582e-03  sqrt(2/d) 2.582e-03



d = 1000: E||z|| simulated 31.6180   delta-method sqrt(d)(1 - 1/(4d)) = 31.6149
        Var||z|| simulated 0.5025   delta-method 1/2 = 0.5


        cosine similarity of independent vectors: mean +0.00009, sd 0.03173, 1/sqrt(d) = 0.03162


### Problem L2.5 — Lasso Is a Laplace Prior (and Ridge Is a Gaussian One)

**Statement.** Show that MAP estimation with a Laplace prior on the weights yields $L_1$ regularization, with a Gaussian prior yields $L_2$, and explain why only the former produces exact zeros.

**Intuition.** The MAP objective is the negative log-likelihood plus the negative log-prior, so the prior's shape at the origin becomes the penalty's shape at the origin.

**Solution**

*Step 1 — write the MAP objective.* With Gaussian likelihood noise of variance $\sigma^2$ and an independent prior $\pi(w)$,

$$
\hat w = \arg\max_w\left[\ln L(w) + \ln\pi(w)\right] = \arg\min_w\left[\frac{1}{2\sigma^2}\lVert y - Xw\rVert^2 - \ln\pi(w)\right].
$$

*Step 2 — insert the Laplace prior.* $\pi(w) = \prod_j\frac{1}{2b}e^{-\lvert w_j\rvert/b}$ gives $-\ln\pi(w) = \frac1b\sum_j\lvert w_j\rvert + c$, so the objective is

$$
\frac{1}{2\sigma^2}\lVert y - Xw\rVert^2 + \frac1b\lVert w\rVert_1 \quad\Longrightarrow\quad \lambda_{\text{lasso}} = \frac{\sigma^2}{b}.
$$

*Step 3 — insert the Gaussian prior.* $\pi(w) = \prod_j\mathcal{N}(0,\tau^2)$ gives $-\ln\pi(w) = \frac{1}{2\tau^2}\lVert w\rVert_2^2 + c$, i.e. ridge with $\lambda_{\text{ridge}} = \sigma^2/\tau^2$.

*Step 4 — explain the exact zeros.* The Laplace density has a *kink* at $0$: $\lvert w\rvert$ is not differentiable there and its subgradient is the whole interval $[-1,1]$. The stationarity condition for coordinate $j$ becomes $\left\lvert x_j^{\top}(y-Xw)\right\rvert \le \lambda$, satisfiable *with* $w_j = 0$ over a whole range of data — the soft-thresholding solution $\hat w_j = \operatorname{sign}(z_j)\left(\lvert z_j\rvert-\lambda\right)^{+}$. The Gaussian prior is smooth at $0$ with zero derivative there, so ridge shrinkage $\hat w_j = z_j/(1+\lambda)$ never reaches exactly $0$.

$$
\boxed{\text{Laplace prior} \Rightarrow L_1 \text{ (sparse, } \lambda = \sigma^2/b); \qquad \text{Gaussian prior} \Rightarrow L_2 \text{ (shrunk, } \lambda = \sigma^2/\tau^2)}
$$

*Key takeaway:* Sparsity is a property of the prior's *shape at the origin* — a density kink, not merely a heavier tail.

In [16]:
lam_pen = 0.8
soft = lambda z: np.sign(z) * np.maximum(np.abs(z) - lam_pen, 0.0)
ridge = lambda z: z / (1 + lam_pen)

zs = np.array([-2.0, -0.5, 0.0, 0.3, 0.8, 1.5])
print(f"z          {zs}")
print(f"lasso      {soft(zs)}")
print(f"ridge      {ridge(zs)}")
print(f"exact zeros: lasso {int(np.sum(soft(zs) == 0))} of {zs.size},  ridge {int(np.sum(ridge(zs) == 0))} of {zs.size}")

for z0 in (0.3, 0.8, 1.5):
    obj_l1 = lambda w: 0.5 * (w - z0) ** 2 + lam_pen * abs(w)
    obj_l2 = lambda w: 0.5 * (w - z0) ** 2 + 0.5 * lam_pen * w**2
    w1 = optimize.minimize_scalar(obj_l1, bounds=(-5, 5), method="bounded").x
    w2 = optimize.minimize_scalar(obj_l2, bounds=(-5, 5), method="bounded").x
    print(f"z0={z0}: numerical L1 argmin {w1:+.6f} (soft-threshold {soft(z0):+.6f});  L2 argmin {w2:+.6f} (z/(1+lam) {ridge(z0):+.6f})")
    assert abs(w1 - soft(z0)) < 1e-4 and abs(w2 - ridge(z0)) < 1e-4

z          [-2.  -0.5  0.   0.3  0.8  1.5]
lasso      [-1.2 -0.   0.   0.   0.   0.7]
ridge      [-1.1111 -0.2778  0.      0.1667  0.4444  0.8333]
exact zeros: lasso 4 of 6,  ridge 1 of 6
z0=0.3: numerical L1 argmin -0.000001 (soft-threshold +0.000000);  L2 argmin +0.166667 (z/(1+lam) +0.166667)
z0=0.8: numerical L1 argmin -0.000000 (soft-threshold +0.000000);  L2 argmin +0.444444 (z/(1+lam) +0.444444)
z0=1.5: numerical L1 argmin +0.700000 (soft-threshold +0.700000);  L2 argmin +0.833333 (z/(1+lam) +0.833333)


### Problem L2.6 — He Initialization From a Variance Calculation

**Statement.** For a layer $y = \sum_{i=1}^{n}w_ix_i$ followed by ReLU, with $w_i$ i.i.d. zero-mean of variance $\mathrm{Var}(w)$ and $x_i$ i.i.d. symmetric with variance $\mathrm{Var}(x)$, derive the initialization scale that keeps activation variance constant across depth.

**Intuition.** Variance multiplies layer by layer, so the initialization must make the per-layer gain exactly one or the signal decays or explodes geometrically.

**Solution**

*Step 1 — propagate variance through the linear map.* Weights and inputs are independent with zero-mean weights, so all cross terms vanish:

$$
\mathrm{Var}(y) = \sum_{i=1}^{n}\mathrm{Var}(w_ix_i) = n\,\mathrm{Var}(w)\,E\left[x^2\right] = n\,\mathrm{Var}(w)\,\mathrm{Var}(x).
$$

*Step 2 — pass through the ReLU.* If $y$ is symmetric about $0$ then $a = \max(y,0)$ satisfies

$$
E\left[a^2\right] = \int_0^{\infty}y^2f(y)\,dy = \tfrac12 E\left[y^2\right] = \tfrac12\mathrm{Var}(y),
$$

because the integrand is even. Each ReLU layer halves the second moment.

*Step 3 — impose the fixed point.* Requiring $E\left[a_{\ell+1}^2\right] = E\left[a_\ell^2\right]$ gives

$$
\tfrac12 n\,\mathrm{Var}(w) = 1 \implies \mathrm{Var}(w) = \frac{2}{n_{\text{in}}},
$$

i.e. He initialization $w \sim \mathcal{N}\left(0, 2/n_{\text{in}}\right)$. Xavier/Glorot uses $1/n_{\text{in}}$ and is correct for symmetric activations such as $\tanh$, which do not halve the variance.

*Step 4 — quantify the cost of getting it wrong.* If the per-layer variance gain is $g \ne 1$, the activation *scale* after $L$ layers is $g^{L/2}$. At $L = 50$ and $g = 0.8$ this is $0.8^{25} = 3.78\times10^{-3}$; reaching $10^{-3}$ takes $L = 2\ln(10^{-3})/\ln(0.8) \approx 62$ layers. Either way the signal decays geometrically — the vanishing-signal problem that principled initialization, and later normalization layers, removed.

$$
\boxed{\mathrm{Var}(w) = \frac{2}{n_{\text{in}}} \text{ for ReLU (He)}; \qquad \frac{1}{n_{\text{in}}} \text{ for symmetric activations (Xavier)}; \qquad 0.8^{25} = 3.78\times10^{-3}}
$$

*Key takeaway:* Deep-network initialization is a variance-propagation calculation; the factor $2$ is literally the ReLU keeping half the mass.

In [17]:
print(f"gain 0.8 over L=50 layers: 0.8^25 = {0.8**25:.6e}  (not 1e-3)")
print(f"depth at which the scale reaches 1e-3: L = 2 ln(1e-3)/ln(0.8) = {2*log(1e-3)/log(0.8):.2f}")
assert abs(0.8**25 - 3.7779e-3) < 1e-6

width, depth, batch = 512, 40, 4096
for name, gain in [("He   2/n_in", 2.0), ("Xavier 1/n_in", 1.0)]:
    a = np.abs(rng.standard_normal((batch, width)))          # post-ReLU input scale
    a *= 1.0 / sqrt(np.mean(a**2))
    scales = []
    for _ in range(depth):
        W = rng.standard_normal((width, width)) * sqrt(gain / width)
        a = np.maximum(a @ W, 0.0)
        scales.append(sqrt(np.mean(a**2)))
    print(f"{name:<14}: RMS activation after 1, 10, 40 layers = {scales[0]:.4f}, {scales[9]:.4f}, {scales[-1]:.4e}")

print(f"\nReLU halving check: E[max(Z,0)^2]/E[Z^2] = {np.mean(np.maximum(rng.standard_normal(4_000_000), 0)**2):.5f}  (exact 0.5)")

gain 0.8 over L=50 layers: 0.8^25 = 3.777893e-03  (not 1e-3)
depth at which the scale reaches 1e-3: L = 2 ln(1e-3)/ln(0.8) = 61.91


He   2/n_in   : RMS activation after 1, 10, 40 layers = 0.9875, 1.2050, 9.0835e-01


Xavier 1/n_in : RMS activation after 1, 10, 40 layers = 0.6808, 0.0255, 1.0352e-06

ReLU halving check: E[max(Z,0)^2]/E[Z^2] = 0.50041  (exact 0.5)


## L3 — Challenge Proofs

### Problem L3.1 — Maxwell's Theorem: Rotational Symmetry Forces the Gaussian

**Statement.** Suppose $X_1,\ldots,X_d$ ($d \ge 2$) are independent, each with a continuous positive density, and the joint density depends only on $\lVert\mathbf x\rVert$. Prove each $X_i$ is $\mathcal{N}(0,\sigma^2)$.

**Intuition.** Independence makes the log joint density a *sum* over coordinates while isotropy makes it a *function of the sum of squares*; only a linear function can be both.

**Solution**

*Step 0 — set up.* Independence gives $f_{\mathbf X}(\mathbf x) = \prod_{i=1}^d f_i(x_i)$, and the isotropy hypothesis gives $f_{\mathbf X}(\mathbf x) = g\left(\lVert\mathbf x\rVert^2\right)$ for some $g \gt 0$. Hence

$$
\prod_{i=1}^{d}f_i(x_i) = g\left(x_1^2 + \cdots + x_d^2\right).
$$

*Step 1 — all marginals coincide.* The joint density depends on $\mathbf x$ only through $\lVert\mathbf x\rVert$, and a coordinate swap preserves $\lVert\mathbf x\rVert$, so it leaves the right-hand side unchanged and therefore leaves the left-hand side unchanged. (A transposition is orthogonal with determinant $-1$ — a reflection, not a rotation — but the hypothesis is invariance under *any* map preserving the norm, so this is legitimate; composing the transposition with a sign flip of one coordinate does give a rotation if one prefers to stay inside $SO(d)$.) Evaluating the swapped identity at points with all but two coordinates fixed gives $f_i \equiv f$ for all $i$.

*Step 2 — reduce to a Cauchy equation.* Take logarithms and set $\phi(u) = \ln f\left(\sqrt u\right)$ for $u \ge 0$, $\psi = \ln g$:

$$
\sum_{i=1}^{d}\phi(u_i) = \psi\left(\sum_{i=1}^{d}u_i\right), \qquad u_i = x_i^2 \ge 0 .
$$

Fix $u_3 = \cdots = u_d = 0$ and write $c = (d-2)\phi(0)$. Then $\phi(u) + \phi(v) + c = \psi(u+v)$ for all $u,v \ge 0$; setting $v = 0$ gives $\psi(u) = \phi(u) + \phi(0) + c$. Substituting eliminates $\psi$:

$$
\phi(u) + \phi(v) = \phi(u+v) + \phi(0).
$$

Put $h(u) = \phi(u) - \phi(0)$; then $h(u+v) = h(u) + h(v)$ with $h(0) = 0$.

*Step 3 — solve the Cauchy equation.* $\phi$ is continuous because $f$ is continuous and positive, so the continuous solutions are exactly the linear ones: $h(u) = -\alpha u$ for a constant $\alpha$. Therefore

$$
\ln f(x) = \phi\left(x^2\right) = \phi(0) - \alpha x^2 \implies f(x) = Ce^{-\alpha x^2}.
$$

*Step 4 — normalize.* Integrability forces $\alpha \gt 0$, and $\int_{\mathbb R} Ce^{-\alpha x^2}dx = 1$ gives $C = \sqrt{\alpha/\pi}$. Writing $\alpha = \frac{1}{2\sigma^2}$ yields $f = \mathcal{N}(0,\sigma^2)$; symmetry about $0$ is automatic since $f$ depends on $x^2$. $\blacksquare$

$$
\boxed{\text{independent components} + \text{rotational invariance} \implies X_i \sim \mathcal{N}\left(0,\sigma^2\right)}
$$

*Key takeaway:* The Gaussian is not merely a convenient bell curve; it is the *only* law reconciling independence with isotropy — which is why Maxwell could derive gas velocities from symmetry alone, with no dynamics.

In [18]:
def joint_ratio(f, pts):
    """max relative spread of the product density over points of equal norm."""
    vals = np.array([np.prod(f(np.asarray(p))) for p in pts])
    return (vals.max() - vals.min()) / vals.mean()


r = 1.7
ring = [(r, 0.0), (0.0, r), (r / sqrt(2), r / sqrt(2)), (r * 0.6, r * sqrt(1 - 0.36))]

gauss = lambda x: np.exp(-(x**2) / 2) / sqrt(2 * pi)
laplace = lambda x: np.exp(-np.abs(x)) / 2
logistic = lambda x: np.exp(-x) / (1 + np.exp(-x)) ** 2

for name, f in [("Gaussian", gauss), ("Laplace", laplace), ("Logistic", logistic)]:
    print(f"{name:<9}: relative spread of the product density on the circle of radius {r} = {joint_ratio(f, ring):.3e}")

assert joint_ratio(gauss, ring) < 1e-14
assert joint_ratio(laplace, ring) > 0.1 and joint_ratio(logistic, ring) > 0.01

alpha = optimize.brentq(lambda a: integrate.quad(lambda x: x**2 * sqrt(a / pi) * np.exp(-a * x**2), -30, 30)[0] - 1.0, 0.05, 5)
print(f"solving Var = 1 inside the family C e^(-alpha x^2): alpha = {alpha:.10f}, expected 1/(2 sigma^2) = 0.5")
assert abs(alpha - 0.5) < 1e-8

Gaussian : relative spread of the product density on the circle of radius 1.7 = 5.548e-16
Laplace  : relative spread of the product density on the circle of radius 1.7 = 6.737e-01
Logistic : relative spread of the product density on the circle of radius 1.7 = 3.348e-02
solving Var = 1 inside the family C e^(-alpha x^2): alpha = 0.5000000000, expected 1/(2 sigma^2) = 0.5


### Problem L3.2 — Independence of the Sample Mean and Sample Variance

**Statement.** For $X_1,\ldots,X_n$ i.i.d. $\mathcal{N}(\mu,\sigma^2)$, prove $\bar X$ and $S^2 = \frac{1}{n-1}\sum_i\left(X_i-\bar X\right)^2$ are independent, with $\bar X \sim \mathcal{N}\left(\mu,\sigma^2/n\right)$ and $\frac{(n-1)S^2}{\sigma^2}\sim\chi^2_{n-1}$, and deduce the $t$ statistic.

**Intuition.** Rotating a standard Gaussian vector leaves it standard Gaussian, so choose the rotation whose first coordinate *is* the mean; everything else is then automatically independent of it.

**Solution**

*Step 1 — reduce to standard form.* Set $Z_i = (X_i-\mu)/\sigma$, so $\mathbf Z \sim \mathcal{N}(0,I_n)$; both claims are invariant under this affine map.

*Step 2 — rotate.* Choose an orthogonal $Q$ whose first row is $\frac{1}{\sqrt n}\mathbf 1^{\top}$ (Gram-Schmidt supplies the rest) and set $\mathbf Y = Q\mathbf Z$. The standard Gaussian density $(2\pi)^{-n/2}e^{-\lVert z\rVert^2/2}$ depends only on $\lVert z\rVert$ and $\lvert\det Q\rvert = 1$, so $\mathbf Y \sim \mathcal{N}(0,I_n)$: the $Y_i$ are again i.i.d. standard normal.

*Step 3 — identify the pieces.* By construction $Y_1 = \frac{1}{\sqrt n}\sum_iZ_i = \sqrt n\,\bar Z$. Orthogonality preserves norms, so

$$
\sum_{i=1}^{n}Y_i^2 = \sum_{i=1}^{n}Z_i^2 \implies \sum_{i=2}^{n}Y_i^2 = \sum_{i=1}^{n}Z_i^2 - n\bar Z^2 = \sum_{i=1}^{n}\left(Z_i-\bar Z\right)^2 .
$$

*Step 4 — conclude independence and the two laws.* $\bar Z$ is a function of $Y_1$ alone while $\sum_i(Z_i-\bar Z)^2$ is a function of $(Y_2,\ldots,Y_n)$ alone, and the $Y_i$ are independent. Hence

$$
\bar X \sim \mathcal{N}\left(\mu,\frac{\sigma^2}{n}\right), \qquad \frac{(n-1)S^2}{\sigma^2} = \sum_{i=2}^{n}Y_i^2 \sim \chi^2_{n-1}, \qquad \bar X \perp S^2 . \qquad \blacksquare
$$

*Step 5 — assemble the $t$ statistic.* With $Z = \frac{\bar X-\mu}{\sigma/\sqrt n}\sim\mathcal{N}(0,1)$ independent of $W = \frac{(n-1)S^2}{\sigma^2}\sim\chi^2_{n-1}$,

$$
T = \frac{\bar X-\mu}{S/\sqrt n} = \frac{Z}{\sqrt{W/(n-1)}} \sim t_{n-1} .
$$

The $\sigma$ cancels, which is the entire point: inference about $\mu$ becomes possible without knowing $\sigma$, at the cost of heavier tails. The $n-1$ degrees of freedom are the one dimension consumed by estimating $\bar X$, and that geometric fact is also why $S^2$ divides by $n-1$ to be unbiased.

$$
\boxed{\bar X \perp S^2, \quad \frac{(n-1)S^2}{\sigma^2}\sim\chi^2_{n-1}, \quad \frac{\bar X - \mu}{S/\sqrt n}\sim t_{n-1}}
$$

*Key takeaway:* This independence holds **only** for the Gaussian; it is a corollary of rotational invariance, and it is what makes classical $t$-inference exact rather than asymptotic.

In [19]:
n, mu, sigma = 8, 3.0, 2.0
reps = 300_000

Xs = rng.normal(mu, sigma, size=(reps, n))
xbar, S2 = Xs.mean(axis=1), Xs.var(axis=1, ddof=1)
W = (n - 1) * S2 / sigma**2
T = (xbar - mu) / np.sqrt(S2 / n)

print(f"corr(xbar, S^2) for normal data      = {np.corrcoef(xbar, S2)[0,1]:+.5f}")
print(f"Var(xbar) {xbar.var():.5f} vs sigma^2/n = {sigma**2/n:.5f}")
print(f"KS of (n-1)S^2/sigma^2 vs chi2_{n-1}:  p = {stats.kstest(W, stats.chi2(n-1).cdf).pvalue:.3f}")
print(f"quantiles of T {np.quantile(T, [0.05, 0.5, 0.95])}   exact t_{n-1} {stats.t(n-1).ppf([0.05, 0.5, 0.95])}")
assert abs(np.corrcoef(xbar, S2)[0, 1]) < 0.01
assert np.max(np.abs(np.quantile(T, [0.05, 0.5, 0.95]) - stats.t(n - 1).ppf([0.05, 0.5, 0.95]))) < 0.05

Ys = rng.exponential(1.0, size=(reps, n))
print(f"\ncorr(xbar, S^2) for exponential data = {np.corrcoef(Ys.mean(axis=1), Ys.var(axis=1, ddof=1))[0,1]:+.5f}")
print("Non-Gaussian data destroys the independence: the property is a Gaussian one.")
assert np.corrcoef(Ys.mean(axis=1), Ys.var(axis=1, ddof=1))[0, 1] > 0.3

corr(xbar, S^2) for normal data      = -0.00083
Var(xbar) 0.49913 vs sigma^2/n = 0.50000
KS of (n-1)S^2/sigma^2 vs chi2_7:  p = 0.746
quantiles of T [-1.8908 -0.0008  1.8894]   exact t_7 [-1.8946  0.      1.8946]

corr(xbar, S^2) for exponential data = +0.69331
Non-Gaussian data destroys the independence: the property is a Gaussian one.


### Problem L3.3 — Deriving Every Maximum-Entropy Family at Once

**Statement.** Show that maximizing $H(f) = -\int f\ln f$ subject to $\int f = 1$ and $\int T_j(x)f(x)\,dx = \tau_j$ yields an exponential family, and specialize to obtain the Uniform, Exponential, Normal and Laplace laws.

**Intuition.** Entropy is strictly concave and the constraints are linear, so a single stationarity condition pins the optimum, and it is always an exponential of the constraint functions.

**Solution**

*Step 1 — form the Lagrangian on the support $\mathcal X$.*

$$
\mathcal{L}[f] = -\int f\ln f - \nu_0\left(\int f - 1\right) - \sum_j\nu_j\left(\int T_jf - \tau_j\right).
$$

*Step 2 — set the functional derivative to zero.*

$$
\frac{\delta\mathcal L}{\delta f(x)} = -\ln f(x) - 1 - \nu_0 - \sum_j\nu_jT_j(x) = 0 \implies f(x) = \exp\left(-1-\nu_0-\sum_j\nu_jT_j(x)\right),
$$

so, absorbing the constant into a normalizer,

$$
f^{\star}(x) = \frac{1}{Z(\boldsymbol\nu)}\exp\left(-\sum_j\nu_jT_j(x)\right)
$$

— an **exponential family** with sufficient statistics $T_j$ and natural parameters $-\nu_j$.

*Step 3 — upgrade the stationary point to a global maximum.* Repeat the KL argument used for the Gaussian: for any feasible $f$,

$$
0 \le D_{\mathrm{KL}}\left(f \parallel f^{\star}\right) = -H(f) - \int f\ln f^{\star},
$$

and $\int f\ln f^{\star} = -\ln Z - \sum_j\nu_j\tau_j$ depends on $f$ only through the fixed constraints. Hence $H(f) \le \ln Z + \sum_j\nu_j\tau_j = H(f^{\star})$ with equality iff $f = f^{\star}$ almost everywhere.

*Step 4 — specialize.*

| Support | Constraints | Resulting $f^{\star}$ |
|---|---|---|
| $[a,b]$ | none beyond normalization | $f \propto 1$: **Uniform**, $H = \ln(b-a)$ |
| $(0,\infty)$ | $E[X] = \mu$ | $f \propto e^{-\nu x}$: **Exponential**$(1/\mu)$, $H = 1 + \ln\mu$ |
| $\mathbb{R}$ | $E[X] = \mu$, $E\left[(X-\mu)^2\right] = \sigma^2$ | $f \propto e^{-\nu_2(x-\mu)^2}$: **Normal**, $H = \tfrac12\ln\left(2\pi e\sigma^2\right)$ |
| $\mathbb{R}$ | $E\lvert X-\mu\rvert = b$ | $f \propto e^{-\nu\lvert x-\mu\rvert}$: **Laplace**$(\mu,b)$, $H = 1+\ln(2b)$ |
| $(0,\infty)$ | $E[X] = \mu$, $E[\ln X]$ fixed | $f \propto x^{\alpha-1}e^{-\lambda x}$: **Gamma** |

$$
\boxed{f^{\star}(x) \propto \exp\left(-\sum_j\nu_jT_j(x)\right): \text{ constraints choose the sufficient statistics, which choose the family}}
$$

*Key takeaway:* The canonical continuous families are one theorem — each is the least-committal law consistent with the moments you are willing to assert, which is why exponential families and maximum entropy are the same subject.

In [20]:
def entropy_of(f, a, b):
    return -integrate.quad(lambda x: f(x) * log(f(x)) if f(x) > 1e-300 else 0.0, a, b, limit=400)[0]


# Unit-variance competitors: the Normal must beat every one of them.
competitors = {
    "Normal(0,1)":      (stats.norm(0, 1).pdf, -40, 40),
    "Laplace var 1":    (lambda x: np.exp(-abs(x) * sqrt(2)) * sqrt(2) / 2, -60, 60),
    "Uniform var 1":    (lambda x: 1 / (2 * sqrt(3)) if abs(x) <= sqrt(3) else 0.0, -sqrt(3), sqrt(3)),
    "t_5 scaled var 1": (stats.t(5, scale=sqrt(3 / 5)).pdf, -60, 60),
    "Logistic var 1":   (stats.logistic(scale=sqrt(3) / pi).pdf, -60, 60),
}
H_gauss = 0.5 * log(2 * pi * e)
for name, (f, a, b) in competitors.items():
    H = entropy_of(f, a, b)
    print(f"{name:<18} variance {integrate.quad(lambda x: x*x*f(x), a, b, limit=400)[0]:.4f}   H = {H:.6f}   max = {H_gauss:.6f}")
    assert H <= H_gauss + 1e-8

# The exponential-family form f ~ exp(-nu2 x^2) with Var = 1 must return nu2 = 1/2.
nu2 = optimize.brentq(
    lambda a: integrate.quad(lambda x: x * x * np.exp(-a * x * x), -30, 30)[0] / integrate.quad(lambda x: np.exp(-a * x * x), -30, 30)[0] - 1.0,
    0.05, 5.0,
)
print(f"\nsolving the variance constraint inside exp(-nu2 x^2): nu2 = {nu2:.10f}  (Gaussian value 1/2)")
assert abs(nu2 - 0.5) < 1e-8

for name, (mu_c, H_cf) in {"Exponential(1/mu), mu=2": (2.0, 1 + log(2.0)), "Laplace(0,b), b=1.5": (1.5, 1 + log(2 * 1.5))}.items():
    print(f"{name:<26} closed-form entropy {H_cf:.6f}")

Normal(0,1)        variance 1.0000   H = 1.418939   max = 1.418939
Laplace var 1      variance 1.0000   H = 1.346574   max = 1.418939
Uniform var 1      variance 1.0000   H = 1.242453   max = 1.418939
t_5 scaled var 1   variance 1.0000   H = 1.372090   max = 1.418939
Logistic var 1     variance 1.0000   H = 1.404576   max = 1.418939

solving the variance constraint inside exp(-nu2 x^2): nu2 = 0.5000000000  (Gaussian value 1/2)
Exponential(1/mu), mu=2    closed-form entropy 1.693147
Laplace(0,b), b=1.5        closed-form entropy 2.098612


### Problem L3.4 — A Distribution Not Determined By Its Moments

**Statement.** Show that the Lognormal has finite moments of all orders yet is **not** determined by them, by exhibiting an explicit family of distinct densities with identical moments.

**Intuition.** Under $x = e^{u}$ the lognormal becomes a Gaussian in $u$, and multiplying by $\sin(2\pi u)$ — a function invisible to every integer shift — kills every polynomial test at once.

**Solution**

*Step 1 — check that all moments exist.* For $X = e^{Y}$ with $Y \sim \mathcal{N}(0,1)$, the $k$-th moment is the Gaussian MGF at $t = k$:

$$
E\left[X^{k}\right] = E\left[e^{kY}\right] = e^{k^2/2} \lt \infty \quad\text{for every } k .
$$

The growth $e^{k^2/2}$ is too fast for Carleman's condition $\sum_k m_{2k}^{-1/(2k)} = \infty$: here $m_{2k}^{-1/(2k)} = e^{-k}$ and $\sum_ke^{-k} \lt \infty$, which is the warning sign.

*Step 2 — perturb.* For $\lvert\epsilon\rvert \le 1$ define, on $x \gt 0$,

$$
f_\epsilon(x) = f_{\text{LN}}(x)\left[1 + \epsilon\sin\left(2\pi\ln x\right)\right], \qquad f_{\text{LN}}(x) = \frac{1}{x\sqrt{2\pi}}e^{-(\ln x)^2/2} .
$$

Non-negativity is immediate from $\lvert\sin\rvert \le 1$.

*Step 3 — show the perturbation is invisible to every moment.* For every integer $k \ge 0$, substitute $u = \ln x$, so $x = e^{u}$ and $dx = e^{u}du$:

$$
\int_0^{\infty}x^{k}f_{\text{LN}}(x)\sin\left(2\pi\ln x\right)dx = \frac{1}{\sqrt{2\pi}}\int_{-\infty}^{\infty}e^{ku}e^{-u^2/2}\sin(2\pi u)\,du .
$$

Complete the square, $ku - \frac{u^2}{2} = \frac{k^2}{2} - \frac{(u-k)^2}{2}$, and shift $w = u-k$:

$$
= \frac{e^{k^2/2}}{\sqrt{2\pi}}\int_{-\infty}^{\infty}e^{-w^2/2}\sin\left(2\pi w + 2\pi k\right)dw = \frac{e^{k^2/2}}{\sqrt{2\pi}}\int_{-\infty}^{\infty}e^{-w^2/2}\sin(2\pi w)\,dw = 0 ,
$$

where $\sin(\theta + 2\pi k) = \sin\theta$ used the integrality of $k$ and the last integral vanishes because its integrand is odd.

*Step 4 — conclude.* Taking $k = 0$ shows each $f_\epsilon$ integrates to $1$, hence is a density; taking general $k$ shows

$$
\int_0^{\infty}x^{k}f_\epsilon(x)\,dx = e^{k^2/2} \quad\text{for all } k \ge 1 \text{ and all } \epsilon \in[-1,1].
$$

So an uncountable family of *distinct* densities shares every moment with the Lognormal. $\blacksquare$

**Why this matters.** The method of moments cannot identify heavy-tailed laws, and MGF-based uniqueness arguments require a finite MGF in a neighbourhood of $0$ — which the Lognormal does not have, since $E\left[e^{tX}\right] = \infty$ for every $t \gt 0$. Sufficient conditions for determinacy are exactly of that type (Carleman's condition, or a finite MGF near the origin).

$$
\boxed{f_\epsilon(x) = f_{\text{LN}}(x)\left[1 + \epsilon\sin\left(2\pi\ln x\right)\right] \text{ has the Lognormal's moments for every } \epsilon}
$$

*Key takeaway:* Moments do not always determine a distribution; the periodicity of $\ln x$ under integer scaling is precisely what lets the perturbation hide from every polynomial test.

In [21]:
def moment_eps(k, eps):
    """int_0^inf x^k f_LN(x) [1 + eps sin(2 pi ln x)] dx, computed after u = ln x."""
    integrand = lambda u: np.exp(k * u) * np.exp(-u * u / 2) / sqrt(2 * pi) * (1 + eps * np.sin(2 * pi * u))
    return integrate.quad(integrand, -40, 40, limit=800)[0]


print(f"{'k':>2} {'exact e^(k^2/2)':>18} {'eps=0':>18} {'eps=0.5':>18} {'eps=1':>18}")
for k in range(5):
    exact = exp(k * k / 2)
    vals = [moment_eps(k, eps) for eps in (0.0, 0.5, 1.0)]
    print(f"{k:>2} {exact:>18.8f} {vals[0]:>18.8f} {vals[1]:>18.8f} {vals[2]:>18.8f}")
    for v in vals:
        assert abs(v - exact) < 1e-8 * max(1.0, exact)

xs = np.linspace(1e-3, 6, 1500)
f_LN = stats.lognorm(1.0).pdf(xs)
for eps in (0.5, 1.0):
    f_eps = f_LN * (1 + eps * np.sin(2 * pi * np.log(xs)))
    print(f"eps = {eps}: min density {f_eps.min():.3e} (non-negative), max sup-distance from lognormal {np.max(np.abs(f_eps - f_LN)):.4f}")
    assert f_eps.min() >= -1e-18

carleman = sum(exp(-k) for k in range(1, 200))
print(f"\nCarleman sum for the lognormal: sum_k m_2k^(-1/2k) = sum_k e^-k = {carleman:.6f} < infinity -> criterion fails")

 k    exact e^(k^2/2)              eps=0            eps=0.5              eps=1
 0         1.00000000         1.00000000         1.00000000         1.00000000
 1         1.64872127         1.64872127         1.64872127         1.64872127
 2         7.38905610         7.38905610         7.38905610         7.38905610
 3        90.01713130        90.01713130        90.01713130        90.01713130
 4      2980.95798704      2980.95798704      2980.95798704      2980.95798704
eps = 0.5: min density 2.210e-08 (non-negative), max sup-distance from lognormal 0.3190
eps = 1.0: min density 1.452e-08 (non-negative), max sup-distance from lognormal 0.6380

Carleman sum for the lognormal: sum_k m_2k^(-1/2k) = sum_k e^-k = 0.581977 < infinity -> criterion fails
